# Production-Grade Portfolio Optimisation

This notebook upgrades a basic Markowitz workflow into a more robust, practical portfolio construction framework.

## What is improved
- cleaner data handling
- monthly total-return series built from adjusted prices
- covariance shrinkage using **Ledoit-Wolf**
- expected return shrinkage to reduce noise
- realistic optimisation constraints
- support for:
  - minimum variance
  - maximum Sharpe
  - minimum CVaR
- optional turnover penalty and transaction cost drag
- risk contribution breakdown
- rolling walk-forward backtest
- clearer outputs for decision-making

## Important caveat
This is still a research notebook, not an execution engine. It is suitable for robust analysis and prototyping, but live deployment would still need production data feeds, order management integration, and stronger controls around missing data, corporate actions, and compliance rules.

In [ ]:
# If needed, uncomment:
# %pip install yfinance scikit-learn scipy matplotlib pandas numpy

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass
from typing import Dict, Iterable, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf

try:
    import yfinance as yf
except ImportError as exc:
    raise ImportError(
        "yfinance is required for this notebook. Install it with: pip install yfinance"
    ) from exc

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Configuration

Adjust only this section for most use cases.

In [ ]:
@dataclass
class PortfolioConfig:
    tickers: List[str]
    benchmark: str = "SPY"
    start_date: str = "2016-01-01"
    end_date: Optional[str] = None

    frequency: str = "M"              # monthly optimisation data
    risk_free_rate_annual: float = 0.04

    objective: str = "max_sharpe"     # min_variance | max_sharpe | min_cvar
    target_return_annual: Optional[float] = None

    min_weight: float = 0.00
    max_weight: float = 0.15
    l2_reg: float = 0.02              # discourages concentration
    turnover_penalty: float = 0.002   # penalty per 100% turnover
    transaction_cost_bps: float = 10  # drag used in backtests

    cvar_alpha: float = 0.95

    mean_shrink: float = 0.60         # 0=no shrinkage, 1=full shrink to grand mean
    min_history_months: int = 24
    rebalance_every_months: int = 1

    walk_forward_train_months: int = 36
    walk_forward_test_months: int = 1


config = PortfolioConfig(
    tickers=[
        "SHEL.L", "BP.L", "HSBA.L", "BARC.L", "AZN.L", "GSK.L",
        "ULVR.L", "DGE.L", "RIO.L", "NG.L", "REL.L", "LSEG.L"
    ],
    benchmark="SPY",
    start_date="2018-01-01",
    objective="max_sharpe",
    max_weight=0.20,
    target_return_annual=None,
)
config

## 2. Data download and cleaning

This uses adjusted close prices and converts them to monthly total returns.

In [ ]:
def download_adjusted_prices(
    tickers: Iterable[str],
    start_date: str,
    end_date: Optional[str] = None,
) -> pd.DataFrame:
    tickers = list(dict.fromkeys(tickers))
    data = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if data.empty:
        raise ValueError("No data was downloaded. Check tickers and date range.")

    if isinstance(data.columns, pd.MultiIndex):
        frames = {}
        for ticker in tickers:
            if ticker in data.columns.get_level_values(0):
                sub = data[ticker]
                col = "Close" if "Close" in sub.columns else sub.columns[-1]
                frames[ticker] = sub[col]
        prices = pd.DataFrame(frames)
    else:
        # Single ticker case
        col = "Close" if "Close" in data.columns else data.columns[-1]
        prices = data[[col]].rename(columns={col: tickers[0]})

    prices = prices.sort_index().dropna(how="all")
    prices = prices.ffill()

    if prices.empty:
        raise ValueError("No usable price history after cleaning.")

    return prices


def to_periodic_returns(prices: pd.DataFrame, frequency: str = "M") -> pd.DataFrame:
    if frequency.upper() == "M":
        sampled = prices.resample("ME").last()
    elif frequency.upper() == "W":
        sampled = prices.resample("W-FRI").last()
    else:
        raise ValueError("Supported frequencies: 'M' or 'W'.")

    rets = sampled.pct_change().dropna(how="all")
    rets = rets.replace([np.inf, -np.inf], np.nan)
    rets = rets.dropna(axis=1, how="all")
    return rets


def clean_returns(returns: pd.DataFrame, min_history_months: int) -> pd.DataFrame:
    clean = returns.copy()

    # Drop assets with too little history
    enough_obs = clean.notna().sum() >= min_history_months
    clean = clean.loc[:, enough_obs]

    # Fill small gaps conservatively, then drop remaining incomplete rows
    clean = clean.ffill(limit=1)
    clean = clean.dropna(axis=0, how="any")

    if clean.shape[1] < 2:
        raise ValueError("Need at least two assets with enough return history.")

    if clean.shape[0] < min_history_months:
        raise ValueError("Not enough clean history after filtering.")

    return clean


prices = download_adjusted_prices(config.tickers + [config.benchmark], config.start_date, config.end_date)
returns_all = to_periodic_returns(prices, config.frequency)
returns = clean_returns(returns_all[config.tickers], config.min_history_months)
benchmark_returns = returns_all[[config.benchmark]].reindex(returns.index).dropna()

print(f"Prices shape   : {prices.shape}")
print(f"Returns shape  : {returns.shape}")
print(f"Assets kept    : {list(returns.columns)}")
returns.tail()

## 3. Robust estimators

Two key upgrades versus a naïve notebook:

1. **Covariance shrinkage** via Ledoit-Wolf  
2. **Expected return shrinkage** towards the cross-sectional grand mean

That reduces unstable and overly concentrated solutions.

In [ ]:
def annualise_return(monthly_return: float) -> float:
    return (1 + monthly_return) ** 12 - 1

def deannualise_return(annual_return: float) -> float:
    return (1 + annual_return) ** (1 / 12) - 1

def estimate_expected_returns(
    returns: pd.DataFrame,
    shrink: float = 0.60,
) -> pd.Series:
    sample_mu = returns.mean()
    grand_mean = float(sample_mu.mean())
    shrunk_mu = (1 - shrink) * sample_mu + shrink * grand_mean
    return shrunk_mu

def estimate_covariance(returns: pd.DataFrame) -> pd.DataFrame:
    lw = LedoitWolf()
    lw.fit(returns.values)
    cov = pd.DataFrame(lw.covariance_, index=returns.columns, columns=returns.columns)
    return cov

mu_m = estimate_expected_returns(returns, config.mean_shrink)
cov_m = estimate_covariance(returns)

mu_a = mu_m.apply(annualise_return)

display(pd.DataFrame({
    "Expected monthly return": mu_m,
    "Expected annual return": mu_a,
}).sort_values("Expected annual return", ascending=False).round(4))

## 4. Optimiser

Supports:
- `min_variance`
- `max_sharpe`
- `min_cvar`

All objectives include:
- long-only bounds
- weight cap
- optional target return floor
- L2 regularisation
- turnover penalty relative to previous weights

In [ ]:
def portfolio_return(weights: np.ndarray, mu: pd.Series) -> float:
    return float(np.dot(weights, mu.values))

def portfolio_variance(weights: np.ndarray, cov: pd.DataFrame) -> float:
    return float(weights @ cov.values @ weights)

def portfolio_volatility(weights: np.ndarray, cov: pd.DataFrame) -> float:
    return float(np.sqrt(max(portfolio_variance(weights, cov), 0.0)))

def portfolio_cvar(weights: np.ndarray, returns: pd.DataFrame, alpha: float = 0.95) -> float:
    port_rets = returns.values @ weights
    cutoff = np.quantile(port_rets, 1 - alpha)
    tail = port_rets[port_rets <= cutoff]
    if len(tail) == 0:
        return 0.0
    return float(-tail.mean())

def risk_contributions(weights: np.ndarray, cov: pd.DataFrame) -> pd.Series:
    sigma = portfolio_volatility(weights, cov)
    if sigma <= 0:
        return pd.Series(0.0, index=cov.index)
    marginal = cov.values @ weights / sigma
    contrib = weights * marginal
    return pd.Series(contrib, index=cov.index)

def solve_portfolio(
    returns: pd.DataFrame,
    mu: pd.Series,
    cov: pd.DataFrame,
    risk_free_rate_annual: float,
    objective: str = "max_sharpe",
    min_weight: float = 0.0,
    max_weight: float = 0.15,
    target_return_annual: Optional[float] = None,
    l2_reg: float = 0.0,
    turnover_penalty: float = 0.0,
    previous_weights: Optional[pd.Series] = None,
    cvar_alpha: float = 0.95,
) -> pd.Series:
    assets = list(returns.columns)
    n = len(assets)
    rf_m = deannualise_return(risk_free_rate_annual)

    if previous_weights is None:
        previous = np.repeat(1 / n, n)
    else:
        previous = previous_weights.reindex(assets).fillna(0.0).values

    x0 = np.repeat(1 / n, n)
    bounds = [(min_weight, max_weight) for _ in range(n)]

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    if target_return_annual is not None:
        target_m = deannualise_return(target_return_annual)
        constraints.append({"type": "ineq", "fun": lambda w: portfolio_return(w, mu) - target_m})

    def common_penalties(w: np.ndarray) -> float:
        reg = l2_reg * float(np.sum(w ** 2))
        turn = turnover_penalty * float(np.sum(np.abs(w - previous)))
        return reg + turn

    def obj_min_variance(w: np.ndarray) -> float:
        return portfolio_variance(w, cov) + common_penalties(w)

    def obj_max_sharpe(w: np.ndarray) -> float:
        vol = portfolio_volatility(w, cov)
        if vol <= 0:
            return 1e6
        sharpe = (portfolio_return(w, mu) - rf_m) / vol
        return -sharpe + common_penalties(w)

    def obj_min_cvar(w: np.ndarray) -> float:
        return portfolio_cvar(w, returns, cvar_alpha) + common_penalties(w)

    objectives = {
        "min_variance": obj_min_variance,
        "max_sharpe": obj_max_sharpe,
        "min_cvar": obj_min_cvar,
    }
    if objective not in objectives:
        raise ValueError(f"Unsupported objective: {objective}")

    result = minimize(
        objectives[objective],
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-9, "disp": False},
    )

    if not result.success:
        raise RuntimeError(f"Optimisation failed: {result.message}")

    weights = pd.Series(result.x, index=assets)
    weights = weights.clip(lower=0)
    weights = weights / weights.sum()
    return weights


weights = solve_portfolio(
    returns=returns,
    mu=mu_m,
    cov=cov_m,
    risk_free_rate_annual=config.risk_free_rate_annual,
    objective=config.objective,
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    target_return_annual=config.target_return_annual,
    l2_reg=config.l2_reg,
    turnover_penalty=config.turnover_penalty,
    previous_weights=None,
    cvar_alpha=config.cvar_alpha,
)

weights.sort_values(ascending=False)

## 5. Portfolio diagnostics

In [ ]:
def build_portfolio_summary(
    weights: pd.Series,
    returns: pd.DataFrame,
    mu: pd.Series,
    cov: pd.DataFrame,
    risk_free_rate_annual: float,
    alpha: float = 0.95,
) -> Tuple[pd.DataFrame, Dict[str, float]]:
    w = weights.values
    port_ret_m = portfolio_return(w, mu)
    port_vol_m = portfolio_volatility(w, cov)
    port_ret_a = annualise_return(port_ret_m)
    port_vol_a = port_vol_m * np.sqrt(12)

    rf_m = deannualise_return(risk_free_rate_annual)
    sharpe = np.nan if port_vol_m <= 0 else (port_ret_m - rf_m) / port_vol_m
    cvar_m = portfolio_cvar(w, returns, alpha=alpha)

    rc = risk_contributions(w, cov)
    rc_pct = rc / rc.sum() if rc.sum() != 0 else rc

    asset_table = pd.DataFrame({
        "Weight": weights,
        "Weight %": weights * 100,
        "Expected annual return %": mu.apply(annualise_return) * 100,
        "Risk contribution %": rc_pct * 100,
    }).sort_values("Weight", ascending=False)

    stats = {
        "Expected return annual %": port_ret_a * 100,
        "Volatility annual %": port_vol_a * 100,
        "Sharpe ratio": sharpe,
        "CVaR monthly %": cvar_m * 100,
        "Effective number of names": float(1 / np.sum(np.square(w))),
    }
    return asset_table, stats


asset_table, stats = build_portfolio_summary(
    weights, returns, mu_m, cov_m, config.risk_free_rate_annual, config.cvar_alpha
)

display(asset_table.round(3))
pd.Series(stats).round(3)

## 6. Efficient frontier

This is useful for seeing whether your chosen solution sits in a sensible region of the feasible set.

In [ ]:
def efficient_frontier(
    returns: pd.DataFrame,
    mu: pd.Series,
    cov: pd.DataFrame,
    config: PortfolioConfig,
    n_points: int = 25,
) -> pd.DataFrame:
    min_ret = float(mu.min())
    max_ret = float(mu.max())

    # Avoid impossible extremes
    targets = np.linspace(min_ret * 0.8, max_ret * 0.9, n_points)

    rows = []
    for target_m in targets:
        try:
            w = solve_portfolio(
                returns=returns,
                mu=mu,
                cov=cov,
                risk_free_rate_annual=config.risk_free_rate_annual,
                objective="min_variance",
                min_weight=config.min_weight,
                max_weight=config.max_weight,
                target_return_annual=annualise_return(target_m),
                l2_reg=config.l2_reg,
                turnover_penalty=0.0,
                previous_weights=None,
                cvar_alpha=config.cvar_alpha,
            )
            vol_a = portfolio_volatility(w.values, cov) * np.sqrt(12)
            ret_a = annualise_return(portfolio_return(w.values, mu))
            rows.append({"vol_a": vol_a, "ret_a": ret_a})
        except Exception:
            continue

    frontier = pd.DataFrame(rows).drop_duplicates().sort_values("vol_a")
    return frontier


frontier = efficient_frontier(returns, mu_m, cov_m, config)
frontier.head()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(frontier["vol_a"] * 100, frontier["ret_a"] * 100, marker="o")
plt.scatter(
    [stats["Volatility annual %"]],
    [stats["Expected return annual %"]],
    s=80,
    label="Selected portfolio"
)
plt.xlabel("Annual volatility (%)")
plt.ylabel("Expected annual return (%)")
plt.title("Efficient Frontier")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 7. Correlation matrix

In [ ]:
corr = returns.corr()

plt.figure(figsize=(9, 7))
plt.imshow(corr.values, aspect="auto")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.index)), corr.index)
plt.colorbar()
plt.title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.show()

## 8. Weight chart and risk contribution chart

In [ ]:
fig = plt.figure(figsize=(9, 4))
plt.bar(asset_table.index, asset_table["Weight %"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Weight (%)")
plt.title("Optimised Portfolio Weights")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(9, 4))
plt.bar(asset_table.index, asset_table["Risk contribution %"])
plt.xticks(rotation=60, ha="right")
plt.ylabel("Risk contribution (%)")
plt.title("Risk Contribution by Asset")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Walk-forward backtest

This is the most important upgrade versus a purely in-sample notebook.

Process:
- train on a rolling window
- optimise on the training sample
- hold for the next period
- include transaction cost drag

In [ ]:
def walk_forward_backtest(
    returns: pd.DataFrame,
    config: PortfolioConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    weights_history = []

    prev_weights = pd.Series(0.0, index=returns.columns)

    train = config.walk_forward_train_months
    test = config.walk_forward_test_months
    step = config.rebalance_every_months

    for start in range(0, len(returns) - train - test + 1, step):
        train_slice = returns.iloc[start : start + train]
        test_slice = returns.iloc[start + train : start + train + test]

        mu = estimate_expected_returns(train_slice, config.mean_shrink)
        cov = estimate_covariance(train_slice)

        try:
            w = solve_portfolio(
                returns=train_slice,
                mu=mu,
                cov=cov,
                risk_free_rate_annual=config.risk_free_rate_annual,
                objective=config.objective,
                min_weight=config.min_weight,
                max_weight=config.max_weight,
                target_return_annual=config.target_return_annual,
                l2_reg=config.l2_reg,
                turnover_penalty=config.turnover_penalty,
                previous_weights=prev_weights,
                cvar_alpha=config.cvar_alpha,
            )
        except Exception:
            continue

        turnover = float(np.abs(w - prev_weights).sum())
        cost = turnover * config.transaction_cost_bps / 10000.0

        realised = float((test_slice.values @ w.values).sum()) - cost

        rows.append({
            "date": test_slice.index[-1],
            "portfolio_return": realised,
            "turnover": turnover,
            "cost": cost,
        })
        weights_history.append(pd.DataFrame({
            "date": test_slice.index[-1],
            "asset": w.index,
            "weight": w.values,
        }))

        prev_weights = w.copy()

    perf = pd.DataFrame(rows).set_index("date")
    weights_hist = pd.concat(weights_history, ignore_index=True) if weights_history else pd.DataFrame()

    if not perf.empty:
        perf["equity_curve"] = (1 + perf["portfolio_return"]).cumprod()

    return perf, weights_hist


perf, weights_hist = walk_forward_backtest(returns, config)
perf.tail()

In [ ]:
def compute_backtest_stats(perf: pd.DataFrame, benchmark_returns: Optional[pd.DataFrame] = None) -> pd.Series:
    if perf.empty:
        return pd.Series(dtype=float)

    r = perf["portfolio_return"]
    ann_ret = (1 + r.mean()) ** 12 - 1
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    sharpe = np.nan if ann_vol == 0 else (ann_ret - config.risk_free_rate_annual) / ann_vol

    cum = (1 + r).cumprod()
    running_max = cum.cummax()
    drawdown = cum / running_max - 1

    stats = {
        "Backtest annual return %": ann_ret * 100,
        "Backtest annual vol %": ann_vol * 100,
        "Backtest Sharpe": sharpe,
        "Max drawdown %": drawdown.min() * 100,
        "Average turnover %": perf["turnover"].mean() * 100,
        "Average cost bps": perf["cost"].mean() * 10000,
    }

    if benchmark_returns is not None and not benchmark_returns.empty:
        bench = benchmark_returns.reindex(perf.index).dropna().iloc[:, 0]
        aligned = perf.loc[bench.index, "portfolio_return"]
        if len(aligned) > 1:
            active = aligned - bench
            te = active.std(ddof=1) * np.sqrt(12)
            ir = np.nan if te == 0 else active.mean() * 12 / te
            stats["Tracking error %"] = te * 100
            stats["Information ratio"] = ir

    return pd.Series(stats)


bt_stats = compute_backtest_stats(perf, benchmark_returns)
bt_stats.round(3)

In [ ]:
if not perf.empty:
    plt.figure(figsize=(9, 5))
    plt.plot(perf.index, perf["equity_curve"], label="Strategy")
    plt.title("Walk-Forward Equity Curve")
    plt.ylabel("Growth of £1")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.bar(perf.index, perf["turnover"] * 100)
    plt.title("Turnover by Rebalance")
    plt.ylabel("Turnover (%)")
    plt.grid(axis="y", alpha=0.3)
    plt.show()
else:
    print("Backtest did not produce any observations. Try a longer history or looser constraints.")

## 10. Current recommended portfolio

This is the clean view most people want to export into a deck or IC memo.

In [ ]:
final_output = asset_table.copy()
final_output["Weight %"] = final_output["Weight %"].round(2)
final_output["Expected annual return %"] = final_output["Expected annual return %"].round(2)
final_output["Risk contribution %"] = final_output["Risk contribution %"].round(2)

summary = pd.Series(stats).round(2)

display(final_output[["Weight %", "Expected annual return %", "Risk contribution %"]])
display(summary)

## 11. Optional extensions

Natural next upgrades:
- sector and factor exposure constraints
- beta neutrality or benchmark-relative optimisation
- Black-Litterman expected returns
- liquidity and ADV constraints
- tax lots and realised gains
- cardinality constraints
- regime switching
- Monte Carlo stress testing
- integration with execution and post-trade monitoring

For a real investment process, the most important disciplines are:
1. robust data
2. realistic constraints
3. out-of-sample validation
4. stability under perturbation
5. governance over overrides